### Practica
Esta notebook tiene por objetivo ver cómo funciona la mutiplicación de matrices para generar los attention_scores.

In [16]:
import numpy as np
import torch
import torch.nn.functional as F

In [26]:
np.random.seed(0)

In [2]:
# Dimensiones
d_model = 4
d_ff = 5 # "Me gusta hacer levantamiento olímpico"
d_k = 3
d_v = 3

In [3]:
# Embeddings random
me_ = np.random.rand(d_model)
gusta_ = np.random.rand(d_model)
hacer_ = np.random.rand(d_model)
levantamiento_ = np.random.rand(d_model)
olimpico_ = np.random.rand(d_model)

In [4]:
print(me_)

[0.6769084  0.1292528  0.16361336 0.71692157]


In [6]:
for word in [me_, gusta_, hacer_, levantamiento_, olimpico_]:
    print(word)

[0.6769084  0.1292528  0.16361336 0.71692157]
[0.71529415 0.37417872 0.26718621 0.66254906]
[0.43126335 0.35821482 0.14465678 0.4358626 ]
[0.71756338 0.0481957  0.88571476 0.84466252]
[0.18436443 0.46289425 0.50587989 0.06490926]


In [7]:
# Posiciones random
pos_1 = np.array([0.1, 0.2, 0.3, 0.4])
pos_2 = np.array([0.2, 0.3, 0.4, 0.5])
pos_3 = np.array([0.3, 0.4, 0.5, 0.6])
pos_4 = np.array([0.4, 0.5, 0.6, 0.7])
pos_5 = np.array([0.5, 0.6, 0.7, 0.8])

In [8]:
# Embeddings con posiciones
me_pos = me_ + pos_1
gusta_pos = gusta_ + pos_2
hacer_pos = hacer_ + pos_3
levantamiento_pos = levantamiento_ + pos_4
olimpico_pos = olimpico_ + pos_5

In [9]:
print(me_pos)

[0.7769084  0.3292528  0.46361336 1.11692157]


In [10]:
# Matrices
w_q = np.random.rand(d_model, d_k) * 0.1
w_k = np.random.rand(d_model, d_k) * 0.1
w_v = np.random.rand(d_model, d_v) * 0.1

In [11]:
me_query = me_pos @ w_q
me_key = me_pos @ w_k
me_value = me_pos @ w_v

In [12]:
gusta_query = gusta_pos @ w_q
gusta_key = gusta_pos @ w_k
gusta_value = gusta_pos @ w_v

In [13]:
hacer_query = hacer_pos @ w_q
hacer_key = hacer_pos @ w_k
hacer_value = hacer_pos @ w_v

In [14]:
levantamiento_query = levantamiento_pos @ w_q
levantamiento_key = levantamiento_pos @ w_k
levantamiento_value = levantamiento_pos @ w_v

In [15]:
olimpico_query = olimpico_pos @ w_q
olimpico_key = olimpico_pos @ w_k
olimpico_value = olimpico_pos @ w_v

In [ ]:
# Similitudes → solo para la palabra "me"
me_me_alpha = me_query @ me_key.T
me_gusta_alpha = me_query @ gusta_key.T
me_hacer_alpha = me_query @ hacer_key.T
me_levantamiento_alpha = me_query @ levantamiento_key.T
me_olimpico_alpha = me_query @ olimpico_key.T

alphas = np.array([me_me_alpha, me_gusta_alpha, me_hacer_alpha, me_levantamiento_alpha, me_olimpico_alpha])
alphas_softmax = F.softmax(torch.tensor(alphas), dim=0).numpy()

In [20]:
print(alphas_softmax)

[0.19619923 0.19936856 0.19852514 0.20444438 0.20146268]


In [21]:
print(alphas_softmax.sum())

1.0


In [23]:
me_attention = np.array([
    alphas_softmax[0] * me_value +
    alphas_softmax[1] * gusta_value +
    alphas_softmax[2] * hacer_value +
    alphas_softmax[3] * levantamiento_value +
    alphas_softmax[4] * olimpico_value
])

In [25]:
me_attention.shape

(1, 3)

In [24]:
print(me_attention)

[[0.15653409 0.09892971 0.28712915]]


### Matrices

In [30]:
# Entrada ya procesada con encodings posicionales
X = np.array([me_pos,
              gusta_pos,
              hacer_pos,
              levantamiento_pos,
              olimpico_pos]) 
X.shape

(5, 4)

In [31]:
X

array([[0.7769084 , 0.3292528 , 0.46361336, 1.11692157],
       [0.91529415, 0.67417872, 0.66718621, 1.16254906],
       [0.73126335, 0.75821482, 0.64465678, 1.0358626 ],
       [1.11756338, 0.5481957 , 1.48571476, 1.54466252],
       [0.68436443, 1.06289425, 1.20587989, 0.86490926]])

In [33]:
Q = X @ w_q
K = X @ w_k
V = X @ w_v

In [34]:
print(Q)

[[0.20141323 0.07993046 0.15100774]
 [0.23718247 0.10478365 0.18049162]
 [0.20955462 0.09648487 0.16438981]
 [0.32637058 0.16831309 0.25816965]
 [0.22542737 0.13398953 0.18768239]]


In [35]:
alphas_matrix = Q @ K.T
alphas_softmax_matrix = F.softmax(torch.tensor(alphas_matrix), dim=0).numpy()

print(alphas_softmax_matrix)

[[0.19727423 0.19645321 0.19664297 0.19524016 0.19590434]
 [0.1994668  0.19929842 0.19933179 0.19905381 0.1991811 ]
 [0.19815572 0.19759523 0.19772947 0.19675891 0.19722182]
 [0.20517815 0.20674446 0.20635513 0.20914066 0.20778309]
 [0.1999251  0.19990869 0.19994063 0.19980646 0.19990964]]


In [36]:
# Ahora multiplicamos las atenciones por los valores
attentions = alphas_softmax_matrix @ V
print(attentions)

[[0.15298561 0.09680198 0.28091147]
 [0.15540285 0.09831172 0.28530021]
 [0.15395504 0.09740735 0.28267139]
 [0.16175123 0.10227615 0.29682459]
 [0.15591668 0.09863134 0.28623053]]
